# 05 · Selección del modelo ganador y registro
**Regla de decisión fijada antes de mirar la prueba:** entre los 3 modelos ajustados, son *elegibles* los que quedan dentro de una desviación entre pliegues del mejor PR-AUC en CV. Entre los elegibles gana el de **menor costo de negocio en CV** (desempate: tiempo y explicabilidad). El segundo de esa lista es el *challenger*.
**Hiperparámetros:** si el tuning no mejoró más que la desviación entre pliegues (ver `04`), los finalistas usan sus valores por defecto (más simples y sin el sesgo de elegir el mejor de 30).
El conjunto de prueba se usa **una sola vez** (sección 3), con el ganador y el segundo.
Ejecutar con *Restart* y luego *Run All*.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, mlflow
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.inspection import permutation_importance
from lightgbm import LGBMClassifier
from src.datos import cargar_obt
from src.pipeline import *
from src.experimento import *
sns.set_theme(style="whitegrid")
df = agregar_features(cargar_obt())
train, test = split_temporal(df, 0.2)
X, y = train[FEATURES], train[TARGET]
Xte, yte = test[FEATURES], test[TARGET]            # NO se usa hasta la sección 3
FECHA_CORTE = df[FECHA].max().isoformat()
configurar()
FABRICA = {
    "Embeddings+LogReg":  lambda: construir_pipeline(LogisticRegression(max_iter=1000), texto="embeddings"),
    "LogisticRegression": lambda: construir_pipeline(LogisticRegression(max_iter=1000)),
    "LightGBM":           lambda: construir_pipeline(LGBMClassifier(random_state=SEED, verbose=-1)),
}
FAMILIA = {"Embeddings+LogReg": "estado_del_arte", "LogisticRegression": "lineal_regularizado", "LightGBM": "boosting"}

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.12/site-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


## 1. Elección con evidencia de CV (sin usar la prueba)

In [2]:
exp = mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO)
runs = mlflow.search_runs([exp.experiment_id])
cand = runs[runs["tags.fase"] == "candidato"].set_index("tags.mlflow.runName")
t = cand.loc[[f"candidato-{n}" for n in FABRICA],
             ["metrics.cv_pr_auc_mean","metrics.cv_pr_auc_std","metrics.costo_cv_umbral","metrics.brecha_pr_auc","metrics.segundos_entrenamiento"]]
t.columns = ["pr_auc_cv","std_cv","costo_cv","brecha","seg_entren"]; t.index = [i.replace("candidato-","") for i in t.index]
mejor = t["pr_auc_cv"].idxmax()
elegibles = t[t["pr_auc_cv"] >= t.loc[mejor,"pr_auc_cv"] - t.loc[mejor,"std_cv"]].sort_values("costo_cv")
if len(elegibles) < 2: elegibles = t.sort_values("pr_auc_cv", ascending=False)
CHAMPION, CHALLENGER = elegibles.index[0], elegibles.index[1]
display(t.round(4)); print("Elegibles:", list(elegibles.index)); print("CHAMPION:", CHAMPION, "| CHALLENGER:", CHALLENGER)

,pr_auc_cv,std_cv,costo_cv,brecha,seg_entren
Embeddings+LogReg,0.9461,0.0145,0.1284,0.0091,1.6454
LogisticRegression,0.9434,0.0176,0.1559,0.0234,0.7122
LightGBM,0.9405,0.0150,0.1562,0.0447,1.6889


Elegibles: ['Embeddings+LogReg', 'LogisticRegression', 'LightGBM']
CHAMPION: Embeddings+LogReg | CHALLENGER: LogisticRegression


### ¿La diferencia entre ganador y segundo supera la variación entre pliegues? (comparación pareada por pliegue)

In [3]:
cvc = validar_cv(FABRICA[CHAMPION](), X, y); cvh = validar_cv(FABRICA[CHALLENGER](), X, y)
dif = np.array(cvc["va_pr"]) - np.array(cvh["va_pr"])
display(pd.DataFrame({CHAMPION: cvc["va_pr"], CHALLENGER: cvh["va_pr"], "dif": dif}, index=range(1,6)).round(4))
print(f"Dif. media = {dif.mean():.4f} | desv. de la diferencia = {dif.std():.4f} | desv. entre pliegues del ganador = {np.std(cvc['va_pr']):.4f}")
print("¿Supera la variación entre pliegues?", abs(dif.mean()) > np.std(cvc["va_pr"]))

,Embeddings+LogReg,LogisticRegression,dif
1,0.9604,0.9630,-0.0026
2,0.9353,0.9303,0.0049
3,0.9371,0.9281,0.0090
4,0.9667,0.9669,-0.0002
5,0.9313,0.9287,0.0026


Dif. media = 0.0027 | desv. de la diferencia = 0.0040 | desv. entre pliegues del ganador = 0.0145
¿Supera la variación entre pliegues? False


## 2. Corridas finales y Model Registry
Se entrena con el train completo y se evalúa en la prueba (una vez) con el umbral elegido en CV. Se registra `resenas-bancos-ec-modelo` con alias `champion` y `challenger`.

In [4]:
cli = mlflow.MlflowClient()
try: cli.delete_registered_model(NOMBRE_MODELO)       # evita versiones duplicadas al re-ejecutar
except Exception: pass
limpiar_fase("final")
final = {}
for rol, nombre in [("champion", CHAMPION), ("challenger", CHALLENGER)]:
    final[rol] = correr(f"final-{rol}-{nombre}", FABRICA[nombre](), "final", FAMILIA[nombre], X, y,
                        {"modelo": nombre, "hiperparametros": "por_defecto"}, FECHA_CORTE,
                        tags={"rol": rol, "modelo": nombre}, prueba=(Xte, yte), registrar=NOMBRE_MODELO, alias=rol)
for rol in final:
    mv = cli.get_model_version_by_alias(NOMBRE_MODELO, rol)
    print(rol, "→ versión", mv.version, "| run_id", mv.run_id)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1396.78it/s]
/usr/local/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/17 02:20:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's objec

🏃 View run final-champion-Embeddings+LogReg at: http://mlflow:5000/#/experiments/2/runs/1056d73032ae49bd83ab59977be1b7eb
🧪 View experiment at: http://mlflow:5000/#/experiments/2


/usr/local/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/17 02:20:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during de

🏃 View run final-challenger-LogisticRegression at: http://mlflow:5000/#/experiments/2/runs/17ac948ece274a2f92de6769521c6af2
🧪 View experiment at: http://mlflow:5000/#/experiments/2
champion → versión 1 | run_id 1056d73032ae49bd83ab59977be1b7eb
challenger → versión 2 | run_id 17ac948ece274a2f92de6769521c6af2


## 3. Resultados en la prueba (hold-out temporal) y métrica de negocio

In [5]:
cols = ["cv_pr_auc_mean","cv_pr_auc_std","test_pr_auc","test_roc_auc","test_f1","test_precision","test_recall","umbral_aplicado","test_costo","test_costo_050"]
tt = pd.DataFrame([{k: v for k, v in r.items() if k != "_modelo"} for r in final.values()]).set_index("nombre")
display(tt[cols].round(4))

,cv_pr_auc_mean,cv_pr_auc_std,test_pr_auc,test_roc_auc,test_f1,test_precision,test_recall,umbral_aplicado,test_costo,test_costo_050
nombre,,,,,,,,,,
final-champion-Embeddings+LogReg,0.9461,0.0145,0.8910,0.9496,0.8600,0.7804,0.9576,0.22,0.1700,0.2484
final-challenger-LogisticRegression,0.9434,0.0176,0.8949,0.9494,0.8623,0.7955,0.9413,0.28,0.1892,0.3157


In [6]:
# Referencias en la prueba (mismo costo: falso negativo = 5, falso positivo = 1)
def ref(nombre, pred):
    return {"modelo": nombre, "costo": costo_negocio(yte, pred), "precision": float(((pred==1)&(yte==1)).sum()/max(pred.sum(),1)), "recall": float(((pred==1)&(yte==1)).sum()/(yte==1).sum())}
filas = [ref("revisar todas (todo = negativa)", np.ones(len(yte), int)),
         ref("no revisar ninguna", np.zeros(len(yte), int)),
         ref("regla de negocio (léxico)", ReglaNegocio().predict(Xte))]
for rol, r in final.items():
    pred = (r["_modelo"].predict_proba(Xte)[:,1] >= r["umbral_aplicado"]).astype(int)
    filas.append(ref(f"{rol}: {r['nombre']}", pred))
display(pd.DataFrame(filas).set_index("modelo").round(4))

,costo,precision,recall
modelo,,,
revisar todas (todo = negativa),0.6468,0.3532,1.0000
no revisar ninguna,1.7661,0.0000,0.0000
regla de negocio (léxico),0.6403,0.8234,0.6660
champion: final-champion-Embeddings+LogReg,0.1700,0.7804,0.9576
challenger: final-challenger-LogisticRegression,0.1892,0.7955,0.9413


### Sensibilidad al costo del error (umbral re-elegido en CV para cada razón FN:FP)

In [7]:
p_te = final["champion"]["_modelo"].predict_proba(Xte)[:,1]
filas = []
for r in (3, 5, 10):
    u, _ = mejor_umbral(cvc["oof_y"], cvc["oof_p"], c_fn=r)
    pred = (p_te >= u).astype(int)
    filas.append({"FN:FP": f"{r}:1", "umbral": u, "costo modelo": costo_negocio(yte, pred, c_fn=r),
                  "costo revisar todas": costo_negocio(yte, np.ones(len(yte), int), c_fn=r),
                  "costo regla": costo_negocio(yte, ReglaNegocio().predict(Xte), c_fn=r),
                  "costo no revisar": costo_negocio(yte, np.zeros(len(yte), int), c_fn=r)})
display(pd.DataFrame(filas).set_index("FN:FP").round(4))

,umbral,costo modelo,costo revisar todas,costo regla,costo no revisar
FN:FP,,,,,
3:1,0.29,0.1380,0.6468,0.4044,1.0597
5:1,0.22,0.1700,0.6468,0.6403,1.7661
10:1,0.21,0.2429,0.6468,1.2302,3.5323


### Desglose del campeón: por banco y sobre textos no vistos en el train

In [8]:
r = final["champion"]; pred = (p_te >= r["umbral_aplicado"]).astype(int)
filas = []
for b, g in test.assign(p=p_te, pred=pred).groupby("banco"):
    fila = {"banco": b, "n": len(g), "% negativa real": g[TARGET].mean()*100}
    if g[TARGET].nunique() == 2 and len(g) >= 30:
        fila.update(pr_auc=average_precision_score(g[TARGET], g["p"]), recall=((g.pred==1)&(g[TARGET]==1)).sum()/(g[TARGET]==1).sum(),
                    costo=costo_negocio(g[TARGET], g.pred))
    filas.append(fila)
display(pd.DataFrame(filas).set_index("banco").round(3))
nuevo = ~Xte[TEXTO].isin(set(X[TEXTO])).values
print(f"Textos no vistos en train: {nuevo.mean()*100:.1f}% de la prueba | PR-AUC en ellos: {average_precision_score(yte[nuevo], p_te[nuevo]):.4f} | costo: {costo_negocio(yte[nuevo], pred[nuevo]):.4f}")

,n,% negativa real,pr_auc,recall,costo
banco,,,,,
Banco Bolivariano,82,54.878,0.917,1.000,0.159
Banco Guayaquil,1081,34.043,0.856,0.943,0.199
Banco Pichincha,2611,34.584,0.892,0.953,0.171
Banco del Austro,30,26.667,0.986,1.000,0.100
Banco del Pacífico,6,83.333,NaN,NaN,NaN
Cooperativa JEP,44,50.000,0.878,0.955,0.295
Diners Club (blu),313,23.642,0.914,0.986,0.089
Produbanco,173,62.428,0.954,1.000,0.110


Textos no vistos en train: 64.0% de la prueba | PR-AUC en ellos: 0.8975 | costo: 0.2107


## 4. Costo y explicabilidad

In [9]:
c = tt[["segundos_entrenamiento", "ms_inferencia_por_resena"]].round(4)
display(c)

,segundos_entrenamiento,ms_inferencia_por_resena
nombre,,
final-champion-Embeddings+LogReg,1.8674,9.7220
final-challenger-LogisticRegression,0.9505,0.0177


In [10]:
# Challenger (LogReg con TF-IDF): palabras que más empujan hacia 'negativa' / 'no negativa'
m = final["challenger"]["_modelo"]
if CHALLENGER == "LogisticRegression":
    coef = pd.Series(m.named_steps["modelo"].coef_[0], index=m.named_steps["prep"].get_feature_names_out()).sort_values()
    print("Más 'no negativa':\n", coef.head(10).round(2).to_string(), "\n\nMás 'negativa':\n", coef.tail(10).round(2).to_string())

Más 'no negativa':
 texto__excelente              -4.60
texto__exelente               -3.14
texto__rapida                 -3.00
texto__facil                  -2.92
texto__gracias                -2.67
texto__genial                 -2.53
texto__ok                     -2.41
texto__pero                   -2.40
texto__excelente aplicacion   -2.40
texto__buena aplicacion       -2.37 

Más 'negativa':
 texto__asco         2.57
texto__ingresar     2.66
texto__terrible     2.76
texto__peor         2.78
texto__porqueria    2.82
texto__lenta        3.05
texto__mal          3.05
texto__pesima       3.52
texto__pesimo       3.82
texto__no           5.45


In [11]:
# Campeón: importancia por variable (permutación sobre la prueba, solo para explicar; no se usa para elegir)
pi = permutation_importance(final["champion"]["_modelo"], Xte, yte, scoring="average_precision", n_repeats=3, random_state=SEED)
display(pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False).round(4).rename("caída de PR-AUC al permutar").to_frame())

,caída de PR-AUC al permutar
comentario_limpio,0.4369
n_neg_lexico,0.0129
log_n_palabras,0.0049
banco,0.0032
version_app_desconocida,0.0026
es_solo_simbolos,-0.0001


## 5. Exportar la evidencia: `docs/corridas.csv`

In [12]:
runs = mlflow.search_runs([exp.experiment_id])
runs.to_csv("/work/docs/corridas.csv", index=False)
print(runs.groupby("tags.fase").size().rename("corridas"))

tags.fase
baseline         2
candidato        5
final            2
tuning          90
tuning_padre     3
Name: corridas, dtype: int64
